In [27]:
from data import *

import torch
import numpy as np
import matplotlib.pyplot as plt

from utils import *
from tqdm import tqdm

In [ ]:
# generate data
d1 = 1000
d2 = 100
r = 1
p = 2.0 / d2

M = get_random_matrix(d1, d2, r) / np.sqrt(d1)
observed_M, masks = get_uniformly_random_samples(M, p)

non_zero_row = ~np.all(masks == 0, axis=1)
observed_M = observed_M[non_zero_row]
masks = masks[non_zero_row]
M = M[non_zero_row]

In [ ]:
import numpy as np

# Example matrices
matrix1 = np.array([[1, 0, 0],
                    [0, 0, 0],
                    [3, 5, 0]])

matrix2 = np.array([[2, 0, 0],
                    [0, 1, 0],
                    [3, 0, 4]])

# Check if non-zero elements are in the same locations
same_nonzero_locations = np.array_equal((matrix1 != 0), (matrix2 != 0))

print(same_nonzero_locations)

In [ ]:
#eps = 0.01

cov_observe_M =  observed_M.T @ observed_M
cov_observe_count = (observed_M == 0).T @ (observed_M == 0)
diag_cov = np.diag( np.diag(cov_observe_M) )

np.count_nonzero(cov_observe_M)

In [31]:
cov_observe_count = (1 * (observed_M != 0)).T @ (1 * (observed_M != 0))
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
T = cov_observe_M / (cov_observe_count/d1)

In [ ]:
T_masks = np.nonzero(T)
MTM = M.T @ M

mask_err = T[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err) / np.linalg.norm(MTM, 'fro'))

In [ ]:
T_p = (1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov)

mask_err_Tp = T_p[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err_Tp) / np.linalg.norm(MTM, 'fro'))

In [ ]:
device = 'cuda:0'
M = torch.from_numpy(M).to(device)
observed_M = torch.from_numpy(observed_M).to(device)
T = torch.from_numpy(T).to(device)
MTM = torch.from_numpy(MTM).to(device)
print(M.shape)

In [ ]:
# impute missing values from rank-r SVD corresponding to masks
train_losses = []
err_estimates = []

epochs = 30
#lr = 0.05
X = T
T_masks = 1 * (T != 0)
for i in range(epochs):
    U, D, Vt = torch.linalg.svd(X)
    D[r:] = 0
    X_update = U @ torch.diag(D) @ Vt

    X = T * T_masks + X_update * (1 - T_masks)

    err = MTM - X
    loss = (err**2).mean()
    train_losses.append(loss.item())
    relative_err = torch.norm(err, 'fro') / torch.norm(MTM, 'fro') 
    err_estimates.append(relative_err.item())
    #print(relative_err)

plt.figure(figsize=(5, 3))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot(err_estimates, label='Err Estimation')
plt.xlabel('Epoch')
plt.ylabel('Err')
plt.title('Err Estimation')
plt.legend()

$U\times V = M$ 

In order to solve $AX=B$, we have 

$
X=U^T, \\
A=V^T, \\
B=M^T
$

Then $V^TU^T=M^T$


In [ ]:

error_list = []
missing_num = 0
total_num = 0
skip = 0
result_list_missing = []
result_list_total = []
use_reg = False
U = []

_, S, V = top_r_svd(X, r)
for i in tqdm(range(M.shape[0])):
    """
    M_row: 1*d2
    V: r*d2
    user_vector: 1*r
    """
    M_row = M[i]
    non_zero_indices = M_row.nonzero(as_tuple=True)[0]

    #non_zero_indices = torch.tensor(range(M_row.shape[0]))

    observed_idx = []
    missing_idx = []
    for non_zero_idx in non_zero_indices:
        if masks[i][non_zero_idx]:
            observed_idx.append(non_zero_idx.item())
        else:
            missing_idx.append(non_zero_idx.item())
    # train mask
    M_row = M_row.unsqueeze(0)
    n = non_zero_indices.shape[0]

    #if len(observed_idx)==0 or len(missing_idx) == 0:
    #    skip +=1
    #    continue
    missing_num += len(missing_idx)
    total_num += len(missing_idx) + len(observed_idx)

    observed_idx = torch.tensor(observed_idx)
    missing_idx = torch.tensor(missing_idx)

    observed_A = V[:, observed_idx].t()
    observed_B = M_row[:, observed_idx].t()
    missing_A = V[:, missing_idx].t()
    missing_B = M_row[:, missing_idx].t()

    if use_reg:
        # Ridge regression
        lambda_reg = 0.001
        I = torch.eye(observed_A.shape[1], device=device) * lambda_reg
        u = torch.linalg.lstsq(observed_A.t() @ observed_A + I, observed_A.t() @ observed_B).solution
    else:
        # Linear regression
        u = torch.linalg.lstsq(observed_A, observed_B).solution   
    U.append(u.t())

    # |AX-B|
    if not (len(observed_idx)==0 or len(missing_idx)) == 0:
        error_list.append(torch.sum((missing_A @ u - missing_B)**2).item())
U = torch.concat(U, dim=0)
print(U.shape)
M_ret = U @ V
print(M_ret)
print(M)
print(M_ret.shape)
err = torch.norm(M-M_ret, p=2) / missing_num
print(err)
ret = np.sqrt(np.sum(error_list)/missing_num)
result_list_missing.append(np.sqrt(np.sum(error_list))/missing_num)
result_list_total.append(np.sqrt(np.sum(error_list))/total_num)
print(np.mean(result_list_missing))
print(np.std(result_list_missing))
print(np.mean(result_list_total))
print(np.std(result_list_total))


